### scrap corpus

In [1]:
!pip install requests
!pip install beautifulsoup4

In [2]:
import requests
from bs4 import BeautifulSoup

def fetch_wikipedia_article(url, output_file="wikipedia.txt"):
    try:
        # កំណត់ headers (ជៀសវាង block)
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }

        # ទាញយក HTML
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse HTML
        soup = BeautifulSoup(response.text, "html.parser")

        # រក content
        content = soup.find("div", {"id": "mw-content-text"})
        if not content:
            print("រកមិនឃើញ content!")
            return

        paragraphs = content.find_all("p")

        # បញ្ចូលអត្ថបទទៅ list
        article_text = []
        for p in paragraphs:
            text = p.get_text().strip()
            if text:
                article_text.append(text)

        # បង្ហាញលទ្ធផល
        print("=== Article Preview ===\n")
        for line in article_text[:5]:  # បង្ហាញតែ 5 paragraph ដំបូង
            print(line + "\n")

        # រក្សាទុកជា file
        with open(output_file, "w", encoding="utf-8") as f:
            for line in article_text:
                f.write(line + "\n\n")

        print(f"\n✅ បានរក្សាទុកជា {output_file}")

    except requests.exceptions.RequestException as e:
        print("Error:", e)


# 🔹 URL អត្ថបទ Wikipedia
url = "https://km.wikipedia.org/wiki/%E1%9E%92%E1%9E%98%E1%9F%92%E1%9E%98%E1%9E%94%E1%9E%91"

# 🔹 ហៅ function
fetch_wikipedia_article(url)

=== Article Preview ===

ធម្មបទ (ធ័ម-មៈ-បត់) (ន.) (បាលី) (Dhammapada) ផ្លូវធម៌, លំអានធម៌ ។ ឈ្មោះគម្ពីរព្រះពុទ្ធសាសនាក្នុងសុត្តន្តបិដកខាងពួកខុទ្ទកនិកាយ ហៅថា ខុទ្ទកនិកាយ ធម្មបទ, ជាពុទ្ធភាសិតសុទ្ធតែជាគាថាទាំងអស់ ហៅថា ធម្មបទគាថា ឬ គាថាធម្មបទ ។ គាថាធម្មបទ អាចត្រូវបានរកឃើញនៅក្នុងគម្ពីរសុត្តន្តបិដក សៀវភៅភាគ៥២ ទំព័រទី ២១ ដល់ទំព័រទី ១១២ នៃគម្ពីរព្រះត្រៃបិដករៀបរៀងជាភាសាខ្មែរ ។

គាថាធម្មបទប្រែ

ភាគទី ១

ព្រះបរមសាស្តាទ្រង់ទេសនាប្រារឰព្រះថេរៈ ឈ្មោះចក្ខុបាល ។ កាលពីព្រះថេរៈឣង្គនេះ លោកនៅជាគ្រហស្ថ មិនទាន់បានសាងផ្នួសនោះ លោកបានទៅស្តាប់នូវព្រះធម៌ទេសនា របស់ព្រះពុទ្ធជាម្ចាស់ហើយ កើតសេចក្តីជ្រះថ្លា ក្នុងព្រះធម៌ទេសនានោះ យ៉ាងក្រៃលែង មានចិត្តប្រាថ្នាចង់បួសជាភិក្ខុ ក្នុងព្រះពុទ្ធសាសនា លុះត្រឡប់មកផ្ទះវិញ ក៏បានប្រគល់ទ្រព្យសម្បត្តិទាំងឣស់ ឲ្យដល់ប្អូនប្រុស របស់ខ្លួនស្រេចហើយ ទើបសុំលាប្អូនទៅបួស ក្នុងសំណាក់ព្រះបរមសាស្តា លុះបួសបាន ៥ ព្រះវស្សាហើយ លោកក៏បានចូលទៅ ក្រាបថ្វាយ បង្គំសុំរៀនព្រះកម្មដ្ឋាន ជាមួយនឹងព្រះសម្មាសម្ពុទ្ធ បន្ទាប់មក ក៏បានក្រាបថ្វាយបង្គំលាព្រះឣង្គទៅ បំពេញសមណធម៌ នៅក្នុងទីកន្លែងមួយដ៏ស្ងាត់ ព្រមជាមួយនឹងភិក្ខុសង

### setup env

In [3]:
!pip install nltk numpy pandas

In [4]:
!pip install khmer-nltk

### cleaning Text

In [5]:
import re
def clean_text(text):
    text = re.sub(r"[^\u1780-\u17FF\s]", "", text)
    return text 
with open("wikipedia.txt","r", encoding="utf-8") as f:
    text = clean_text(f.read())

### khmer tokenizer

In [14]:
from khmernltk import word_tokenize
tokens = word_tokenize(text)

ModuleNotFoundError: No module named 'khmernltk'

### Vocabulary + < UNK >

In [ ]:
from collections import Counter

freq = Counter(tokens)
vocab = {w for w,c in freq.items() if c >= 2}

### splite Dataset(70/10/20)

In [ ]:
n = len(tokens)

train = tokens[:int(0.7*n)]
test = tokens[int(0.8*n):]
val = tokens[int(0.7 * n):int(0.8 * n)]

#cheak verify size
print(f"Train size: {len(train)}")
print(f"Test size: {len(test)}")
print(f"Validation size: {len(val)}")

Train size: 80327
Test size: 22951
Validation size: 11475


### create N-gram

In [ ]:
from collections import Counter

def build_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# build ngrams
train_4 = build_ngrams(train, 4)
train_3 = build_ngrams(train, 3)
train_2 = build_ngrams(train, 2)

# count
count_4 = Counter(train_4)
count_3 = Counter(train_3)
count_2 = Counter(train_2)
count_1 = Counter(train)



### LM1 – Backoff Model

In [ ]:
def backoff_prob(w1, w2, w3, w4):
    if count_3[(w1,w2,w3)] > 0 and count_4[(w1,w2,w3,w4)] > 0:
        return count_4[(w1,w2,w3,w4)] / count_3[(w1,w2,w3)]

    elif count_2[(w2,w3)] > 0 and count_3[(w2,w3,w4)] > 0:
        return count_3[(w2,w3,w4)] / count_2[(w2,w3)]

    elif count_1[w3] > 0 and count_2[(w3,w4)] > 0:
        return count_2[(w3,w4)] / count_1[w3]

    else:
        return count_1[w4] / len(train)
## best model (unsmoothed)

### LM2 – Interpolation + Add-k

In [ ]:
# add-k smoothing
def smoothed_prob(count_ngram, count_prev, V, k=0.01):
    return (count_ngram + k) / (count_prev + k*V)

In [ ]:
# interpolation
lambda1, lambda2, lambda3, lambda4 = 0.25, 0.25, 0.25, 0.25
V = len(vocab)

def interpolated_prob(w1, w2, w3, w4):
    p4 = smoothed_prob(count_4[(w1,w2,w3,w4)], count_3[(w1,w2,w3)], V)
    p3 = smoothed_prob(count_3[(w2,w3,w4)], count_2[(w2,w3)], V)
    p2 = smoothed_prob(count_2[(w3,w4)], count_1[w3], V)
    p1 = smoothed_prob(count_1[w4], len(train), V)
    return lambda1*p4 + lambda2*p3 + lambda3*p2 + lambda4*p1


### Perplexity Evaluation

In [ ]:
import math

def perplexity(model, tokens):
    N = len(tokens)
    log_prob = 0

    for i in range(3, N):
        w1,w2,w3,w4 = tokens[i-3], tokens[i-2], tokens[i-1], tokens[i]
        p = model(w1,w2,w3,w4)

        if p == 0:
            p = 1e-10  

        log_prob += math.log(p)

    return math.exp(-log_prob / N)

In [ ]:
pp_backoff = perplexity(backoff_prob, test)
pp_interpolated = perplexity(interpolated_prob, test)
print(f"Backoff Perplexity: {pp_backoff}")
print(f"Interpolated Perplexity: {pp_interpolated}")

Backoff Perplexity: 73.25934595597384
Interpolated Perplexity: 92.97567907899438


### Text Generation

In [ ]:
import random

def generate(model, seed, length=20):
    result = seed[:]

    for _ in range(length):
        w1,w2,w3 = result[-3:]
        candidates = list(vocab)

        probs = [model(w1,w2,w3,w) for w in candidates]
        next_word = random.choices(candidates, weights=probs)[0]

        result.append(next_word)

    return " ".join(result)

In [ ]:
print(generate(interpolated_prob, ["ខ្ញុំ","សូត្រ","ភាសា"]))

ខ្ញុំ សូត្រ ភាសា ហើយ   ក្រហាយ ទឹកឣ នោះ ហើយ   គឺ សុក្កំ   បដិសន្ធិ ជាបង សេចក្តីសង្ឃឹម នេះ ឈរ បន ពង សម្រាល វេរំ  
